# 实践课：图像分割（第 7 章）

**时长**：约 90 分钟，**下课前提交**本 notebook（保留全部运行输出），文件名 `学号_姓名_图像分割.ipynb`。

**数据**（`data/` 目录，全班同一份，都是真实图像）：

| 文件 | 内容 | 用在 |
|---|---|---|
| `coins.png` | 24 枚古希腊硬币的黑白照片，光照不均 | 任务 1、2 |
| `nuclei.png` | 荧光显微镜下的细胞核，很多粘在一起 | 任务 3 |
| `pet1~4.jpg` + `pet1~4_mask.png` | 4 张猫狗照片 + 人工标注的前景真值（Oxford-IIIT Pet） | 任务 0、4、5、6 |
| `info.json` | 每张宠物图的前景外接矩形 `[x, y, w, h]`（任务 6 用） | |

**任务一览**

| 任务 | 内容 | 要求 | 建议用时 |
|---|---|---|---|
| 0 | IoU 与 Dice | 自己实现 | 5 min |
| 1 | 大津法 Otsu | (A) 调库 + (B) 自己实现 + (C) 对比 | 20 min |
| 2 | 局部阈值 + 形态学：数硬币 | 调用 | 10 min |
| 3 | 距离变换 + 分水岭：分开粘连的细胞核 | 调用 | 10 min |
| 4 | K-means 颜色聚类分割 | (A) 调库 + (B) 自己实现 + (C) 对比 | 25 min |
| 5 | Mean Shift 与 SLIC 超像素 | 调用 | 10 min |
| 6 | 图割 GrabCut + 方法总对比 | 调用 | 10 min |

凡是标了 `# TODO` 的地方都需要你来写；每个任务最后的 **思考** 用一两句话回答在对应的 Markdown 单元格里。

In [ ]:
# 环境准备（直接运行）
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.filters import threshold_sauvola
from skimage.segmentation import slic, mark_boundaries, watershed

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti SC',
                                   'Noto Sans CJK SC', 'Noto Sans CJK JP', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
DATA = 'data/'
INFO = json.load(open(DATA + 'info.json'))
print('OpenCV', cv2.__version__)

def show(imgs, titles=None, cmap='gray', size=3.3):
    # 一行显示多张图
    n = len(imgs)
    plt.figure(figsize=(size * n, size))
    for i, im in enumerate(imgs):
        plt.subplot(1, n, i + 1)
        plt.imshow(im, cmap=cmap if np.ndim(im) == 2 else None)
        if titles: plt.title(titles[i], fontsize=11)
        plt.axis('off')
    plt.tight_layout(); plt.show()

def load_pet(i):
    # 返回 RGB 图像 (H,W,3) uint8 和前景真值 (H,W) bool
    img = cv2.cvtColor(cv2.imread(DATA + 'pet%d.jpg' % i), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(DATA + 'pet%d_mask.png' % i, 0) > 127
    return img, gt

def rect_mask(shape, rect):
    x, y, w, h = rect
    m = np.zeros(shape[:2], bool); m[y:y + h, x:x + w] = True
    return m

def pick_foreground(labels, rect):
    # 聚类结果 → 前景掩码：某一类像素落在前景框内的比例，若高于"框占整幅图的面积比例"
    # （即这一类在框内比框外更密集），就认为它属于前景
    inside = rect_mask(labels.shape, rect)
    fg = np.zeros(labels.shape, bool)
    for c in np.unique(labels):
        m = labels == c
        if (m & inside).sum() / m.sum() > inside.mean():
            fg |= m
    return fg

pets = [load_pet(i) for i in range(1, 5)]
show([p[0] for p in pets] + [pets[0][1]], ['pet1', 'pet2', 'pet3', 'pet4', 'pet1 真值'])

## 任务 0　评价指标：IoU 与 Dice（自己实现）

预测掩码 $P$、真值 $G$（都是 bool 数组）：

$$\mathrm{IoU}=\frac{|P\cap G|}{|P\cup G|},\qquad \mathrm{Dice}=\frac{2|P\cap G|}{|P|+|G|}$$

两者都为空时约定为 1。下面的断言全部通过才算完成。

In [ ]:
def iou(pred, gt):
    pred, gt = np.asarray(pred, bool), np.asarray(gt, bool)
    # TODO
    raise NotImplementedError

def dice(pred, gt):
    pred, gt = np.asarray(pred, bool), np.asarray(gt, bool)
    # TODO
    raise NotImplementedError

a = np.array([[1, 1, 0, 0]] * 2, bool); b = np.array([[0, 1, 1, 0]] * 2, bool)
assert abs(iou(a, b) - 1 / 3) < 1e-9 and abs(dice(a, b) - 0.5) < 1e-9
assert iou(a, a) == 1 and iou(a, ~a) == 0 and iou(a & False, a & False) == 1
print('IoU(a,b) = %.4f   Dice(a,b) = %.4f   通过' % (iou(a, b), dice(a, b)))

## 任务 1　大津法 Otsu

在灰度直方图上找阈值 $t$：灰度 $\le t$ 为第 0 类，$>t$ 为第 1 类。设 $p_i$ 为灰度 $i$ 的像素占比，

$$\omega_0(t)=\sum_{i\le t}p_i,\quad \mu(t)=\sum_{i\le t}i\,p_i,\quad \mu_T=\sum_{i}i\,p_i,
\qquad \sigma_B^2(t)=\frac{\big(\mu_T\,\omega_0(t)-\mu(t)\big)^2}{\omega_0(t)\,\big(1-\omega_0(t)\big)}$$

最优阈值 $t^*=\arg\max_t\sigma_B^2(t)$（类间方差最大 ⇔ 类内方差最小）。

### (A) 调库

In [ ]:
coins = cv2.imread(DATA + 'coins.png', 0)
# TODO：用 cv2.threshold 的 THRESH_OTSU 得到阈值 T_cv 和二值图 bin_cv
T_cv, bin_cv = None, None
print('OpenCV 大津阈值 =', T_cv)
show([coins, bin_cv], ['原图', 'OpenCV Otsu, T=%d' % T_cv])

### (B) 自己实现

要求：只用 NumPy，**不许写对 256 个阈值的 Python 循环**——用 `np.cumsum` 一次算出所有 $t$ 的 $\sigma_B^2(t)$。
函数返回 `(t_star, sigma_b2)`，`sigma_b2` 是长度 256 的数组（分母为 0 的位置置 0）。

In [ ]:
def my_otsu(gray):
    # TODO ① 灰度直方图 → 概率 p（长度 256）
    # TODO ② 用 cumsum 求 omega0(t)、mu(t)，以及 mu_T
    # TODO ③ 按公式算 sigma_b2，分母为 0 处置 0
    # TODO ④ t_star = argmax
    raise NotImplementedError

T_my, sb2 = my_otsu(coins)
bin_my = (coins > T_my).astype(np.uint8) * 255
print('自己实现的阈值 =', T_my)

### (C) 对比

1. 在 `coins` 和 4 张宠物图（转灰度）上，两种实现的阈值是否完全相同？二值图有多少像素不同？
2. 画出直方图与 $\sigma_B^2(t)$ 曲线，标出 $t^*$。

In [ ]:
names = ['coins'] + ['pet%d' % i for i in range(1, 5)]
grays = [coins] + [cv2.cvtColor(p[0], cv2.COLOR_RGB2GRAY) for p in pets]
# TODO：对每张图分别求 OpenCV 阈值和自己实现的阈值，并统计二值图中不同像素的个数，打印成表

fig, ax1 = plt.subplots(figsize=(7, 3))
ax1.bar(np.arange(256), np.bincount(coins.ravel(), minlength=256), color='#999', width=1)
ax1.set_xlabel('灰度'); ax1.set_ylabel('像素数')
ax2 = ax1.twinx(); ax2.plot(sb2, 'r'); ax2.set_ylabel('类间方差 $\\sigma_B^2$', color='r')
ax2.axvline(T_my, color='r', ls='--'); ax2.text(T_my + 3, sb2.max() * 0.9, 't*=%d' % T_my, color='r')
plt.title('coins：直方图与类间方差'); plt.tight_layout(); plt.show()
show([coins, bin_my, bin_cv], ['原图', '自己实现 T=%d' % T_my, 'OpenCV T=%d' % T_cv])

**思考 1**：观察 `coins` 的二值图——上方背景和左下角硬币出了什么问题？原因是什么？

（在这里作答）

## 任务 2　局部阈值 + 形态学：数硬币（调用）

1. 用 `skimage.filters.threshold_sauvola(coins, window_size=51, k=0.1)` 得到**逐像素**阈值图 `T_map`，`coins > T_map` 为前景。
2. 先开运算（去小噪点）再闭运算（补小缺口），结构元用 5×5 椭圆 `cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))`。
3. 填洞：`cv2.findContours(..., cv2.RETR_EXTERNAL, ...)` 找外轮廓，再用 `cv2.drawContours(..., -1, 255, -1)` 填实。
4. 用 `cv2.connectedComponentsWithStats` 计数：**面积 < 500 的、碰到图像边框的**连通域不算。

对 Otsu 的二值图走同样的 2–4 步，比较两者数出的硬币个数（真值 24）。

In [ ]:
K5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

def clean_and_count(binary, min_area=500):
    # 返回 (处理后的二值图, 硬币个数)
    # TODO：开运算 → 闭运算 → 填洞 → 连通域计数（去掉小区域与碰边区域）
    raise NotImplementedError

# TODO：Sauvola 二值化得到 bin_sv
bin_sv = None
res_otsu, n_otsu = clean_and_count(bin_cv)
res_sv, n_sv = clean_and_count(bin_sv)
print('Otsu 数出 %d 枚，Sauvola 数出 %d 枚（真值 24）' % (n_otsu, n_sv))
show([bin_sv, res_otsu, res_sv], ['Sauvola 二值', 'Otsu+形态学: %d 枚' % n_otsu, 'Sauvola+形态学: %d 枚' % n_sv])

for ws in [25, 51, 75, 101]:
    b = (coins > threshold_sauvola(coins, window_size=ws, k=0.1)).astype(np.uint8) * 255
    print('window_size=%3d → %d 枚' % (ws, clean_and_count(b)[1]))

**思考 2**：窗口太小（25）为什么会数错？窗口取多大合适，和什么有关？

（在这里作答）

## 任务 3　距离变换 + 分水岭：分开粘连的细胞核（调用）

1. `nuclei.png` 用 Otsu 二值化，3×3 椭圆做一次开运算，数连通域个数 `n_cc`。
2. 距离变换 `cv2.distanceTransform(b, cv2.DIST_L2, 5)`，再 `cv2.GaussianBlur(dist, (0, 0), 1.0)` 平滑。
3. **标记**：距离图的局部极大值——`ds == cv2.dilate(ds, np.ones((7, 7), np.uint8))` 且 `ds > 2`；`cv2.connectedComponents` 给每个标记编号。
4. `skimage.segmentation.watershed(-ds, markers, mask=b > 0)`（对 **负** 距离图灌水），数区域个数 `n_ws`。

In [ ]:
nuc = cv2.imread(DATA + 'nuclei.png', 0)
# TODO 1~4，得到 b、n_cc、cc_lab(连通域标签图)、ds、markers、labels、n_ws
print('连通域 %d 个  →  分水岭 %d 个' % (n_cc, n_ws))

def color_labels(lab, seed=1):
    rng = np.random.RandomState(seed)
    pal = rng.randint(60, 255, (lab.max() + 1, 3)).astype(np.uint8); pal[0] = 0
    return pal[lab]
show([nuc, color_labels(cc_lab), ds, color_labels(labels)],
     ['细胞核', '连通域: %d' % n_cc, '距离变换', '分水岭: %d' % n_ws], cmap='magma', size=3.6)

**思考 3**：把局部极大值的窗口从 7×7 改成 3×3 或 15×15，个数怎么变？分别对应"过分割"还是"欠分割"？

（在这里作答）

## 任务 4　K-means 颜色聚类分割

把每个像素看成 Lab 颜色空间里的一个点（$N\times3$），聚成 $K$ 类：

1. 初始化 $K$ 个中心；2. 每个点分给最近的中心；3. 中心更新为本类均值；4. 重复 2–3 直到分配不再变化。

目标函数（误差平方和）$\mathrm{SSE}=\sum_i\|x_i-c_{z_i}\|^2$。

### (A) 调库

In [ ]:
img1, gt1 = pets[0]
X1 = cv2.cvtColor(img1, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)
# TODO：cv2.kmeans 聚成 K=3 类（attempts=5, cv2.KMEANS_PP_CENTERS，终止条件 (EPS+MAX_ITER, 100, 1e-4)）
# 得到 sse_cv（cv2.kmeans 的第一个返回值）、lab_cv（H×W 的类别图）、cen_cv
print('OpenCV K-means, K=3, SSE = %.4g' % sse_cv)

### (B) 自己实现

要求：分配步骤用**广播**一次算出 $N\times K$ 距离矩阵（不许对像素写循环）；从数据中随机选 $K$ 个点作初始中心（`np.random.RandomState(seed)`）；
某类变空时保留旧中心。返回 `labels`（长度 N）、`centers`（K×3）、`history`（每轮 SSE 的列表）。

In [ ]:
def my_kmeans(X, K, max_iter=100, seed=0):
    # TODO ① 随机选 K 个数据点作初始中心
    # TODO ② 循环：算 N×K 距离 → argmin 分配 → 记录 SSE → 分配不变则停 → 更新中心（空类保留旧中心）
    raise NotImplementedError

lab_my, cen_my, hist_my = my_kmeans(X1, 3, seed=0)
print('自己实现: 迭代 %d 轮, SSE = %.4g' % (len(hist_my), hist_my[-1]))
assert all(a >= b - 1e-6 * a for a, b in zip(hist_my, hist_my[1:])), 'SSE 应该单调不增！'

### (C) 对比

1. $K=2\ldots6$：自己实现（种子 0、1、2 取 SSE 最小的一次）与 OpenCV 的 SSE 对比，画"肘部曲线"。
2. 每个像素换成所属类中心的颜色，显示 $K=2,3,5$ 的分割结果。
3. 用 `pick_foreground` 把 $K=2$ 的聚类结果变成前景掩码，算 4 张宠物图的 IoU。

In [ ]:
def to_rgb(labels, centers, shape):
    lab_img = centers[labels].reshape(shape).astype(np.uint8)
    return cv2.cvtColor(lab_img, cv2.COLOR_LAB2RGB)

iou_km = []
# TODO 1：K=2..6 的 SSE 对比表 + 肘部曲线
# TODO 2：K=2,3,5 的分割可视化
# TODO 3：4 张图 K=2 的前景 IoU，存到列表 iou_km

**思考 4**：pet4（黑白花狗站在绿草里）为什么 K-means 的 IoU 很低？只靠颜色聚类的根本局限是什么？

（在这里作答）

## 任务 5　Mean Shift 与 SLIC 超像素（调用）

1. **Mean Shift**：`cv2.pyrMeanShiftFiltering(bgr, sp, sr)`（`sp` 空间带宽，`sr` 颜色带宽），对 pet1 试 `(sp, sr)` = (10, 20)、(20, 40)、(30, 60)。
2. **SLIC**：`slic(img, n_segments=n, compactness=m)`，用 `mark_boundaries` 显示边界。对 pet2 试 `m` = 1, 10, 40（n=200）和 `n` = 50, 200, 800（m=10）。
3. **超像素能"保住"边界吗？** 让每个超像素整体取它内部多数像素的真值标签，得到的掩码与真值的 IoU 就是"用这组超像素最好能做到的 IoU"。对 pet2 算 n = 50, 200, 800 时的这个值。

In [ ]:
# TODO 1~3

**思考 5**：紧凑度 $m$ 变大时超像素形状怎样变化？为什么超像素常被当作后续分割/分类的"预处理"？

（在这里作答）

## 任务 6　图割 GrabCut + 方法总对比（调用）

GrabCut = 图割（能量 = 区域项 + 边界项，最小割求解）+ 用 GMM 颜色模型迭代更新区域项。用户只需给一个框。

1. 对 4 张宠物图，用 `INFO['rects']['petX']` 作为框调用
   `cv2.grabCut(bgr, mask, rect, bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)`（调用前 `cv2.setRNGSeed(0)`；`bgd`、`fgd` 为 `np.zeros((1, 65), np.float64)`）。
   `mask` 中值为 `cv2.GC_FGD` 或 `cv2.GC_PR_FGD` 的像素为前景。
2. 同样用 `pick_foreground` 规则，给"灰度大津法"（两类）也算 IoU。
3. 汇总成表：Otsu / K-means(K=2) / GrabCut 在 4 张图上的 IoU 与平均值。

In [ ]:
# TODO 1~3，得到列表 iou_otsu、iou_gc
iou_otsu, iou_gc = [], []

print('%-10s' % '方法' + ''.join('%8s' % ('pet%d' % i) for i in range(1, 5)) + '%8s' % '平均')
for name, v in [('Otsu', iou_otsu), ('K-means', iou_km), ('GrabCut', iou_gc)]:
    print('%-10s' % name + ''.join('%8.3f' % x for x in v) + '%8.3f' % np.mean(v))

**思考 6**：GrabCut 比 K-means 好在哪里？（提示：能量函数的两项分别解决了 K-means 的什么问题。）这三种方法都不需要训练数据——要想让机器"知道"什么是猫狗，下一步该怎么做？

（在这里作答）

---
**提交前检查**：从头 `Restart & Run All` 一遍，确认没有报错、所有图和表都显示出来，再保存提交。